In [ ]:
import pandas as pd
df = pd.read_csv("../data/interim/wildfire_clean.csv")

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

mask =  df.CAUSE_CLASS != "Unknown"
x = pd.concat([np.log(df.FIRE_SIZE), df[["LATITUDE", "LONGITUDE"]]], axis=1)[mask]
y = df.CAUSE_CLASS[mask]

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision_macro',
    'recall': 'recall_macro',
    'f1': 'f1_macro',
    'roc_auc': 'roc_auc'
}

X_train, X_test, y_train, y_test = train_test_split(x, y, random_state=37)
def split_sample(n, random_state=None):
    X_train_sample = X_train.sample(n=n, random_state=random_state)
    y_train_sample = y_train.loc[X_train_sample.index]
    X_test_sample = X_test.sample(n=n, random_state=random_state)
    y_test_sample = y_test.loc[X_test_sample.index]
    return X_train_sample, X_test_sample, y_train_sample, y_test_sample

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

X_train_sample, X_test_sample, y_train_sample, y_test_sample = split_sample(10000, random_state=37)

common_C = np.logspace(-4, 4, 10)
common_reg = ['l2', 'l1', 'elasticnet']
common_weight = [None, 'balanced']

param_grid = [
    {
    'clf': [LogisticRegression(random_state=37)],
    'clf__C': common_C,
    'clf__penalty': common_reg,
    'clf__solver': ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
    'clf__class_weight': common_weight
    },
    {
    'clf':[LinearSVC(random_state=37)],
    'clf__C': common_C,
    'clf__penalty': common_reg[:2],
    'clf__loss': ['hinge', 'squared_hinge'],
    'clf__class_weight': common_weight
    }
]

pipeline = Pipeline([("stc", StandardScaler()), ("clf", LogisticRegression())])
searchx = GridSearchCV(pipeline, param_grid, scoring=scoring, refit="accuracy")
searchx.fit(X_train_sample, y_train_sample)

searchx.cv_results_

In [11]:

# def rank_summary(cv_results, clf_col="param_clf"):
#     results_df = pd.DataFrame(cv_results)

#     rank_cols = [col for col in results_df.columns if col.startswith("rank_test_")]
#     score_cols = [col for col in results_df.columns if col.startswith("mean_test_")]

#     ranks = results_df[rank_cols]
#     scores = results_df[score_cols]

#     results_df["model"] = results_df[clf_col].apply(lambda x: type(x).__name__)

#     summary_df = pd.concat([results_df[["model"]], ranks, scores], axis=1)
#     summary_df["total_rank"] = summary_df[rank_cols].sum(axis=1)

#     return summary_df.sort_values(by="total_rank").reset_index(drop=True)


def rank_summary(cv_results, clf_col="param_clf"):
    results_df = pd.DataFrame(cv_results)

    rank_cols = [col for col in results_df.columns if col.startswith("rank_test_")]
    score_cols = [col for col in results_df.columns if col.startswith("mean_test_")]

    ranks = results_df[rank_cols]
    scores = results_df[score_cols]

    results_df["model"] = results_df[clf_col].apply(lambda x: type(x).__name__)
    
    # Create index for joining later
    results_df["model_index"] = range(len(results_df))

    # First DataFrame: summary with ranks and scores
    summary_df = pd.concat([results_df[["model", "model_index"]], ranks, scores], axis=1)
    summary_df["total_rank"] = summary_df[rank_cols].sum(axis=1)
    summary_df = summary_df.sort_values(by="total_rank").reset_index(drop=True)
    
    # Second DataFrame: parameters for each model
    param_cols = [col for col in results_df.columns if col.startswith("param_")]
    params_df = results_df[["model", "model_index"] + param_cols]
    
    # Sort params_df in the same order as summary_df
    params_df = params_df.set_index("model_index").loc[summary_df["model_index"]].reset_index()
    
    # Remove the temporary index columns
    summary_df = summary_df.drop("model_index", axis=1)
    params_df = params_df.drop("model_index", axis=1)
    
    return summary_df, params_df


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from scipy.stats import randint

common_max_depth = np.linspace(1, 20, 10).astype(int).tolist() + [None]
common_min_split = common_min_leaf = [1, 2, 5, 10, 15, 20, 30, 50, 75, 100]
common_min_gain = list(np.linspace(0, 1, 5))
common_estimators = randint.rvs(50, 500, size=10)
common_n_leaves = np.linspace(10, 500, 10, dtype=int)
common_max_leaf = [None, *common_n_leaves]
common_alpha = common_min_frac = common_impurity = np.linspace(0, .1, 10)

param_grid = [
    {
        'clf': [KNeighborsClassifier()],
        'clf__n_neighbors': [*[1, 3, 5, 7, 9, 11, 15], *range(20, 50, 10)],
        'clf__weights': ['uniform', 'distance'],
    },
    {
        'clf': [DecisionTreeClassifier(random_state=37)],
        'clf__max_depth': common_max_depth,
        'clf__min_samples_split': common_min_split,
        'clf__min_samples_leaf': common_min_leaf,
        'clf__min_weight_fraction_leaf': common_min_frac,
        'clf__min_impurity_decrease': common_impurity,
        'clf__max_leaf_nodes': common_max_leaf,
        'clf__ccp_alpha': common_alpha,
        'clf__class_weight': common_weight
    },
    {
        'clf': [RandomForestClassifier(random_state=37)],
        'clf__n_estimators': common_estimators,
        'clf__max_depth': common_max_depth,
        'clf__min_samples_split': common_min_split,
        'clf__min_samples_leaf': common_min_leaf,
        'clf__min_weight_fraction_leaf': common_min_frac,
        'clf__min_impurity_decrease': common_impurity,
        'clf__max_leaf_nodes': common_max_leaf,
        'clf__ccp_alpha': common_alpha,
        'clf__class_weight': common_weight
    }
]

pipeline = Pipeline([("stc", StandardScaler()), ("clf", DecisionTreeClassifier())])
searches = RandomizedSearchCV(pipeline, param_grid, scoring=scoring, refit="accuracy", n_iter=100, random_state=37)
searches.fit(X_train_sample, y_train_sample)

searches.best_score_

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
import lightgbm as lgb, xgboost as xgb

common_learning = np.logspace(-4, -1, 10)
common_regs = np.linspace(0, 10, 5)
common_estimators = np.append(common_estimators, 1000)


In [ ]:

param_grid = [
    {
        'clf': [SVC(random_state=37)],
        'clf__C': common_C,
        'clf__gamma': np.logspace(-5, 1, 10),
        'clf__class_weight': common_weight
    },
    {
        'clf': [AdaBoostClassifier(random_state=37)],
        'clf__n_estimators': common_estimators,
        'clf__learning_rate': common_learning,
        # 'clf__ccp_alpha': common_alpha,
        # 'clf__class_weight': common_weight
    },
    {
        'clf': [GradientBoostingClassifier(random_state=37)],
        'clf__n_estimators': common_estimators,
        'clf__learning_rate': common_learning,
        'clf__loss': ['log_loss', 'deviance', 'exponential'],
        'clf__max_depth': common_max_depth,
        'clf__min_samples_split': common_min_split,
        'clf__min_samples_leaf': common_min_leaf,
        'clf__min_weight_fraction_leaf': common_min_frac,
        # 'clf__num_leaves': common_n_leaves,
        # 'clf__min_split_gain': common_min_gain,
        # 'clf__min_child_weight': common_min_split,
        # 'clf__min_child_samples': common_min_split,
        'clf__min_impurity_decrease': common_impurity,
        'clf__max_leaf_nodes': common_max_leaf,
        'clf__ccp_alpha': common_alpha,
        # 'clf__class_weight': common_weight
    },
    {
        'clf': [lgb.LGBMClassifier(random_state=37)],
        'clf__n_estimators': common_estimators,
        'clf__learning_rate': common_learning,
        'clf__reg_alpha': common_regs,
        'clf__reg_lambda': common_regs,
        'clf__max_depth': common_max_depth,
        # 'clf__min_samples_split': common_min_split,
        # 'clf__min_samples_leaf': common_min_split,
        # 'clf__min_weight_fraction_leaf': common_min_frac,
        'clf__num_leaves': common_n_leaves,
        'clf__min_split_gain': common_min_gain,
        'clf__min_child_weight': common_min_split,
        'clf__min_child_samples': common_min_split,
        # 'clf__min_impurity_decrease': common_impurity,
        # 'clf__max_leaf_nodes': common_max_leaf,
        # 'clf__ccp_alpha': common_alpha,
        'clf__class_weight': common_weight
    },
    {
        'clf': [xgb.XGBClassifier(random_state=37)],
        'clf__n_estimators': common_estimators,
        'clf__learning_rate': common_learning,
        'clf__reg_alpha': common_regs,
        'clf__reg_lambda': common_regs,
        'clf__max_depth': common_max_depth,
        'clf__min_samples_split': common_min_split,
        'clf__min_samples_leaf': common_min_split,
        'clf__min_weight_fraction_leaf': common_min_frac,
        'clf__num_leaves': common_n_leaves,
        'clf__min_split_gain': common_min_gain,
        'clf__min_child_weight': common_min_split,
        'clf__min_child_samples': common_min_split,
        'clf__min_impurity_decrease': common_impurity,
        'clf__max_leaf_nodes': common_max_leaf,
        'clf__ccp_alpha': common_alpha,
        'clf__class_weight': common_weight
    },
]


In [ ]:

pipeline = Pipeline([("stc", StandardScaler()), ("clf", SVC())])

searchx = RandomizedSearchCV(pipeline, param_grid[0], scoring=scoring, refit="accuracy", n_iter=1, random_state=37)
# searchx = GridSearchCV(pipeline, param_grid[0], scoring=scoring, refit="accuracy")
searchx.fit(X_train_sample, y_train_sample)

searchx.cv_results_

In [ ]:
search = RandomizedSearchCV(pipeline, param_grid[1], scoring=scoring, refit="accuracy", n_iter=10, random_state=37)
# search = GridSearchCV(pipeline, param_grid, scoring=scoring, refit="accuracy")
search.fit(X_train_sample, y_train_sample)


In [ ]:
search = RandomizedSearchCV(pipeline, param_grid[2], scoring=scoring, refit="accuracy", n_iter=10, random_state=37)
# search = GridSearchCV(pipeline, param_grid, scoring=scoring, refit="accuracy")
search.fit(X_train_sample, y_train_sample)


In [ ]:

search = RandomizedSearchCV(pipeline, param_grid[3], scoring=scoring, refit="accuracy", n_iter=10, random_state=37)
# search = GridSearchCV(pipeline, param_grid, scoring=scoring, refit="accuracy")
search.fit(X_train_sample, y_train_sample)

In [ ]:


param_grid = [
    {
        'clf': [SVC(random_state=37)],
        'clf__C': common_C,
        'clf__gamma': np.logspace(-5, 1, 10),
        'clf__class_weight': common_weight
    },
    {
        'clf': [AdaBoostClassifier(random_state=37)],
        'clf__n_estimators': common_estimators,
        'clf__learning_rate': common_learning,
        # 'clf__ccp_alpha': common_alpha,
        # 'clf__class_weight': common_weight
    },
    {
        'clf': [GradientBoostingClassifier(random_state=37)],
        'clf__n_estimators': common_estimators,
        'clf__learning_rate': common_learning,
        'clf__loss': ['log_loss', 'deviance', 'exponential'],
        'clf__max_depth': common_max_depth,
        'clf__min_samples_split': common_min_split,
        'clf__min_samples_leaf': common_min_leaf,
        'clf__min_weight_fraction_leaf': common_min_frac,
        # 'clf__num_leaves': common_n_leaves,
        # 'clf__min_split_gain': common_min_gain,
        # 'clf__min_child_weight': common_min_split,
        # 'clf__min_child_samples': common_min_split,
        'clf__min_impurity_decrease': common_impurity,
        'clf__max_leaf_nodes': common_max_leaf,
        'clf__ccp_alpha': common_alpha,
        # 'clf__class_weight': common_weight
    },
    {
        'clf': [lgb.LGBMClassifier(random_state=37)],
        'clf__n_estimators': common_estimators,
        'clf__learning_rate': common_learning,
        'clf__reg_alpha': common_regs,
        'clf__reg_lambda': common_regs,
        'clf__max_depth': common_max_depth,
        # 'clf__min_samples_split': common_min_split,
        # 'clf__min_samples_leaf': common_min_split,
        # 'clf__min_weight_fraction_leaf': common_min_frac,
        'clf__num_leaves': common_n_leaves,
        'clf__min_split_gain': common_min_gain,
        'clf__min_child_weight': common_min_split,
        'clf__min_child_samples': common_min_split,
        # 'clf__min_impurity_decrease': common_impurity,
        # 'clf__max_leaf_nodes': common_max_leaf,
        # 'clf__ccp_alpha': common_alpha,
        'clf__class_weight': common_weight
    },
    {
        'clf': [xgb.XGBClassifier(random_state=37)],
        # 'clf__n_estimators': common_estimators,
        'clf__learning_rate': common_learning,
        'clf__grow_policy': ['depthwise', 'lossguide'],
        'clf__reg_alpha': common_regs,
        'clf__reg_lambda': common_regs,
        'clf__max_delta_step': common_regs,
        'clf__max_depth': common_max_depth,
        # 'clf__min_samples_split': common_min_split,
        # 'clf__min_samples_leaf': common_min_split,
        # 'clf__min_weight_fraction_leaf': common_min_frac,
        'clf__max_leaves': common_n_leaves,
        'clf__num_parallel_tree': common_n_leaves,
        # 'clf__min_split_gain': common_min_gain,
        'clf__min_child_weight': common_min_split,
        # 'clf__min_child_samples': common_min_split,
        # 'clf__min_impurity_decrease': common_impurity,
        # 'clf__max_leaf_nodes': common_max_leaf,
        # 'clf__ccp_alpha': common_alpha,
        'clf__scale_pos_weight': common_min_split[:-2],
        'clf__num_parallel_tree': common_min_split, 
        # monotone_constraints ??weirdasf
        # interaction_constraints combinationsbruh
    },
]

In [ ]:
import numpy as np
mapping = {'Human': 1, 'Natural': 0}
y_train_binary = np.array([mapping[label] for label in y_train_sample])
y_test_binary  = np.array([mapping[label] for label in y_test_sample])

In [ ]:
searchy = RandomizedSearchCV(pipeline, param_grid[4], scoring=scoring, refit="accuracy", n_iter=3, random_state=37)
# searchy = GridSearchCV(pipeline, param_grid[4], scoring=scoring, refit="accuracy")
searchy.fit(X_train_sample, y_train_binary)

searchy.cv_results_

In [ ]:

# scoring = {
#     'accuracy': 'accuracy',
#     'precision': 'precision_macro',
#     'recall': 'recall_macro',
#     'f1': 'f1_macro',
#     'roc_auc': 'roc_auc'
# }

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb

from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
# from bayes_opt import BayesianOptimization

# --- Data Loading ---
# Replace the following with your actual data-loading code:
# For example, if you have a CSV with cleaned data:
# df = pd.read_csv("data/interim/wildfire_clean.csv")
# X = df[['FIRE_SIZE', 'LATITUDE', 'LONGITUDE']]  # adjust features as needed
# y = df['CAUSE_CLASS']  # adjust target column as needed

# Here we assume X and y are already defined.
# For demonstration, we'll use placeholders:
# X, y = ... 
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision_macro',
    'recall': 'recall_macro',
    'f1': 'f1_macro',
    'roc_auc': 'roc_auc'
}

# Split data into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=37)

# --- Define the Pipeline ---
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", lgb.LGBMClassifier(random_state=37))
])

# --- Define the Search Space ---
search_space = {
    "clf__reg_lambda": Real(7.5, 10.0, prior="uniform"),
    "clf__reg_alpha": Real(0.0, 2.5, prior="uniform"),
    "clf__num_leaves": Integer(220, 400),
    "clf__n_estimators": Integer(150, 500),
    "clf__min_split_gain": Real(0.20, 0.30, prior="uniform"),
    "clf__min_child_weight": Integer(2, 5),
    "clf__min_child_samples": Categorical([1, 20]),
    "clf__max_depth": Integer(13, 17),
    "clf__learning_rate": Real(0.13, 0.23, prior="uniform"),
    "clf__class_weight": Categorical(['balanced'])
}

# --- Set Up BayesSearchCV ---
bayes_search = BayesSearchCV(
    estimator=pipeline,
    search_spaces=search_space,
    n_iter=1,           # Number of parameter settings sampled
    scoring='f1_macro',  # Change scoring if needed (e.g., 'recall_macro')
    cv=2,                # 3-fold cross-validation
    n_jobs=-1,
    random_state=37,
    verbose=1
)

# --- Run Bayesian Optimization ---
bayes_search.fit(X_train, y_train)

# --- Output the Results ---
print("Best Parameters:", bayes_search.best_params_)
print("Best CV Score:", bayes_search.best_score_)

# Optionally, evaluate on the test set:
test_score = bayes_search.score(X_test, y_test)
print("Test Score:", test_score)

import joblib

# Save only the best parametersdef build_pipeline(params_path: str = "models/best_lgbm_params.pkl", random_state: int = 37):

joblib.dump(bayes_search.best_params_, "../models/best_lgbm_params.pkl")



SyntaxError: invalid syntax. Perhaps you forgot a comma? (2990918772.py, line 65)

In [55]:
from sklearn.metrics import classification_report

# Predict on the test set using the best estimator
y_pred = bayes_search.best_estimator_.predict(X_test)

# Generate and print the classification report
report = classification_report(y_test, y_pred)
print("Classification Report for Best Hyperparameters:")
print(report)

import joblib

# Save the best estimator to a file
# joblib.dump(bayes_search.best_estimator_, "best_lgbm_model.pkl")
print("Model saved as best_lgbm_model.pkl")


Classification Report for Best Hyperparameters:
              precision    recall  f1-score   support

       Human       0.97      0.86      0.91    444840
     Natural       0.53      0.84      0.65     81147

    accuracy                           0.86    525987
   macro avg       0.75      0.85      0.78    525987
weighted avg       0.90      0.86      0.87    525987

Model saved as best_lgbm_model.pkl


In [ ]:
import pandas as pd
from pycaret.classification import setup, compare_models, tune_model, evaluate_model, finalize_model, predict_model

# --- Prepare Your Data ---
# Assume X_train_sample is your DataFrame with features and y_train_sample is your Series of labels.
# Also, assume X_test is your test set DataFrame (with the same feature columns).

X_train_sample = X_train.sample(n=10000, random_state=37)
y_train_sample = y_train.loc[X_train_sample.index]

X_test_sample = X_test.sample(n=10000, random_state=37)
y_test_sample = y_test.loc[X_test_sample.index]

mapping = {'Human': 1, 'Natural': 0}

# Convert your training labels to binary
y_train_binary = y_train_sample.map(mapping)

# Combine features and target into one DataFrame for PyCaret
train_data = X_train_sample.copy()
train_data['target'] = y_train_binary

# --- Set Up the PyCaret Experiment ---
# You can adjust parameters like normalize, remove_multicollinearity, or feature_selection as needed.
clf_setup = setup(data=train_data,
                  target='target', 
                  session_id=37,
                  fold=3, 
                  normalize=True, 
                  remove_multicollinearity=True,
                  feature_selection=True)

# --- Compare Models ---
# This will train a variety of classifiers and sort them by Recall.
best_model = compare_models(sort='F1')

# --- Tune the Best Model ---
# Fine-tune the best model based on Recall.
tuned_model = tune_model(best_model, optimize='F1')

# --- Evaluate the Tuned Model ---
# This will open an interactive dashboard with various evaluation plots.
evaluate_model(tuned_model)
# --- Finalize the Model ---
final_model = finalize_model(tuned_model)

# --- Predict on Test Data ---
# Ensure that your X_test DataFrame has the same feature columns as your training data.
predictions = predict_model(final_model, data=X_test_sample)
print(predictions.head())


In [57]:
# Assume `model` is your trained classifier and `X_unknown` contains your Unknown cases
import numpy as np
import pandas as pd

# Predict probabilities on the unknown data
probs = model.predict_proba(X_unknown)  # shape: (n_samples, n_classes)
# Assuming the classes are ordered as [Human, Natural]
human_probs = probs[:, 0]  # if "Human" is index 0

# Set a high threshold for reclassifying as Human
threshold = 0.95

# Create a DataFrame for clarity
results = pd.DataFrame({
    'predicted_human_prob': human_probs,
    'predicted_label': model.predict(X_unknown)
})

# Automatically reclassify only those with high confidence in "Human"
results['final_label'] = np.where(results['predicted_human_prob'] >= threshold, 'Human', 'Unknown')

# Optionally, inspect the distribution
print(results.describe())
print(results['final_label'].value_counts())


NameError: name 'model' is not defined

In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.kernel_approximation import RBFSampler
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

df_class0 = df[df.CAUSE_CLASS == "Human"]
df_class1 = df[df.CAUSE_CLASS == "Natural"]

n_samples = min(df.CAUSE_CLASS.value_counts().values)

df_class0_down = df_class0.sample(n=n_samples, random_state=37)
df_class1_down = df_class1.sample(n=n_samples, random_state=37)

# Combine and shuffle the balanced DataFrame
balanced_df = pd.concat([df_class0_down, df_class1_down]).sample(frac=1, random_state=37).reset_index(drop=True)


In [6]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
X = balanced_df[['FIRE_SIZE', 'LATITUDE', 'LONGITUDE']]
y = balanced_df['CAUSE_CLASS']  # assuming 'Human' and 'Natural'
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=37)

common_max_depth = np.linspace(1, 20, 10).astype(int).tolist() + [None]
common_min_split = common_min_leaf = [1, 2, 5, 10, 15, 20, 30, 50, 75, 100]
common_min_gain = list(np.linspace(0, 1, 5))
# common_estimators = randint.rvs(50, 500, size=10)
common_n_leaves = np.linspace(10, 500, 10, dtype=int)
common_max_leaf = [None, *common_n_leaves]
common_alpha = common_min_frac = common_impurity = np.linspace(0, .1, 10)
common_C = np.logspace(-4, 4, 10)
common_learning = np.logspace(-4, -1, 10)
common_regs = np.linspace(0, 10, 5)
# common_estimators = np.append(common_estimators, 1000)

common_reg = ['l2', 'l1', 'elasticnet']
common_weight = [None, 'balanced']

# Pipeline with SGDClassifier (without kernel approximation)
param_grid = [
    {
    'clf': [SGDClassifier(random_state=37)],
    },
]
pipeline_sgd = Pipeline([
    ('scaler', StandardScaler()),
    ('sgd', SGDClassifier(random_state=37))
])

# Parameter grid for plain SGDClassifier
param_grid_sgd = {
    'sgd__eta0': common_learning,
    'sgd__learning_rate': ['optimal', 'adaptive', 'invscaling', 'constant'],
    'sgd__alpha': common_alpha[1:],
    'sgd__penalty': common_reg,
    'sgd__loss': ['hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron'],
    'sgd__power_t': common_min_gain,
    'sgd__class_weight': common_weight
}

# Pipeline 2: StandardScaler + RBFSampler + SGDClassifier
pipeline_kernel = Pipeline([
    ('scaler', StandardScaler()),
    ('rbf', RBFSampler(random_state=37)),
    ('sgd', SGDClassifier(random_state=37))
])


In [8]:

# Parameter grid for SGDClassifier with kernel approximation
param_grid_kernel = {
    'rbf__n_components': [50, 100, 300, 500, 1000, 2000],
    'rbf__gamma': common_alpha,
    'sgd__eta0': common_learning,
    'sgd__learning_rate': ['optimal', 'adaptive', 'invscaling', 'constant'],
    'sgd__alpha': common_alpha[1:],
    'sgd__penalty': common_reg,
    'sgd__loss': ['hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron'],
    'sgd__power_t': common_min_gain,
    'sgd__class_weight': common_weight
}

# Run GridSearchCV for the plain SGD pipeline
grid_search_sgd = RandomizedSearchCV(pipeline_sgd, param_grid_sgd, scoring='f1_macro', random_state=37, n_iter=100)
grid_search_sgd.fit(X_train, y_train)
print("Best score for SGD pipeline:", grid_search_sgd.best_score_)
grid_search_sgd.cv_results_


Best score for SGD pipeline: 0.7550541060807531


{'mean_fit_time': array([0.55716462, 1.11565409, 0.2732626 , 0.26356897, 0.70149436,
        0.26235838, 0.29172821, 0.27636628, 0.23208394, 0.19772525,
        0.32725706, 0.22061362, 0.22313104, 0.38072267, 0.48874154,
        0.25104256, 0.27030272, 0.29465089, 0.29254541, 0.82485976,
        0.19638171, 0.21201124, 0.98624029, 0.21639628, 0.61642671,
        0.27894311, 1.32838659, 0.20681782, 0.97409506, 0.77538395,
        0.69890327, 0.58817253, 1.01752768, 1.01770639, 0.26093469,
        0.20713377, 1.20109806, 0.21441612, 0.25090327, 0.24563012,
        0.57368231, 0.24855251, 0.78422828, 0.26325769, 0.24150867,
        0.22773681, 0.94973216, 0.99751391, 0.23338714, 0.9791738 ,
        0.90583529, 0.22491384, 0.23668604, 0.87275195, 0.24272223,
        0.25839767, 0.24101243, 0.89661264, 0.31873965, 0.23601179,
        0.29007368, 0.2599339 , 0.25515919, 0.31597133, 0.26190577,
        0.55672092, 0.2664052 , 1.13442845, 0.58354149, 0.31902652,
        1.04435697, 0.67905283,

In [ ]:
import joblib
#DONT DUMP IF FOUND BY RSCV THO USE 
joblib.dump(grid_search_sgd.best_params_, "../models/best_sgd_params.pkl")

['../models/best_sgd_params.pkl']

In [10]:

# Run GridSearchCV for the kernel-approximated pipeline
grid_search_kernel = RandomizedSearchCV(pipeline_kernel, param_grid_kernel, scoring='f1_macro', random_state=37, n_iter=10)
grid_search_kernel.fit(X_train, y_train)
print("Best score for Kernel+SGD pipeline:", grid_search_kernel.best_score_)
grid_search_kernel.cv_results_
import joblib
# joblib.dump(grid_search_kernel.best_params_, "../models/best_sgd_params.pkl")

KeyboardInterrupt: 

In [ ]:
def rank_summary(cv_results, model_type="unknown"):
    results_df = pd.DataFrame(cv_results)

    rank_cols = [col for col in results_df.columns if col.startswith("rank_test_")]
    score_cols = [col for col in results_df.columns if col.startswith("mean_test_")]

    # Use the provided model_type instead of trying to extract it from a parameter key.
    results_df["model"] = model_type
    
    # Create an index column for sorting/joining
    results_df["model_index"] = range(len(results_df))

    # Build the summary DataFrame
    summary_df = pd.concat([results_df[["model", "model_index"]], results_df[rank_cols], results_df[score_cols]], axis=1)
    summary_df["total_rank"] = summary_df[rank_cols].sum(axis=1)
    summary_df = summary_df.sort_values(by="total_rank").reset_index(drop=True)

    # Build a parameters DataFrame from all parameter columns
    param_cols = [col for col in results_df.columns if col.startswith("param_")]
    params_df = results_df[["model", "model_index"] + param_cols]
    params_df = params_df.set_index("model_index").loc[summary_df["model_index"]].reset_index()

    # Drop the temporary index column and return
    summary_df = summary_df.drop("model_index", axis=1)
    params_df = params_df.drop("model_index", axis=1)

    return summary_df, params_df


In [ ]:
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision_macro',
    'recall': 'recall_macro',
    'f1': 'f1_macro',
    'roc_auc': 'roc_auc'
}

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.kernel_approximation import RBFSampler
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

# Assume balanced_df is your balanced DataFrame
# Let's split the data in a stratified manner:
X = balanced_df[['FIRE_SIZE', 'LATITUDE', 'LONGITUDE']]
y = balanced_df['CAUSE_CLASS']  # assuming 'Human' and 'Natural'

# For simplicity, use a stratified split to maintain balance in both train and test sets.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=37)

# Pipeline with SGDClassifier (without kernel approximation)
sgd_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SGDClassifier(random_state=37))
])
sgd_pipeline.fit(X_train, y_train)
y_pred_sgd = sgd_pipeline.predict(X_test)
print("SGD Classifier Report:")
print(classification_report(y_test, y_pred_sgd))
print("ROC AUC:", roc_auc_score((y_test=='Human').astype(int), sgd_pipeline.decision_function(X_test)))

# Pipeline with kernel approximation + SGDClassifier
kernel_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rbf', RBFSampler(gamma=1.0, random_state=37)),  # Adjust gamma as needed
    ('clf', SGDClassifier(random_state=37))
])
kernel_pipeline.fit(X_train, y_train)
y_pred_kernel = kernel_pipeline.predict(X_test)
print("\nKernel Approximation + SGD Classifier Report:")
print(classification_report(y_test, y_pred_kernel))
print("ROC AUC:", roc_auc_score((y_test=='Human').astype(int), kernel_pipeline.decision_function(X_test)))


SGD Classifier Report:
              precision    recall  f1-score   support

       Human       0.77      0.72      0.75     20732
     Natural       0.74      0.79      0.76     20732

    accuracy                           0.75     41464
   macro avg       0.76      0.75      0.75     41464
weighted avg       0.76      0.75      0.75     41464

ROC AUC: 0.27087833090109775

Kernel Approximation + SGD Classifier Report:
              precision    recall  f1-score   support

       Human       0.80      0.80      0.80     20732
     Natural       0.80      0.80      0.80     20732

    accuracy                           0.80     41464
   macro avg       0.80      0.80      0.80     41464
weighted avg       0.80      0.80      0.80     41464

ROC AUC: 0.14841141818920095


In [3]:
import pandas as pd, joblib
# import os;
# os.chdir("hejoefs/portfolio/projects/main");
# print(os.getcwd());
from src.utils import assign

df = pd.read_csv("data/interim/wildfire_clean.csv")
print(df.CAUSE_CLASS.value_counts())
model = joblib.load("models/best_lgbm_model.pkl")
mask=df["CAUSE_CLASS"].isin(["Human", "Natural"])
new_data = assign(df.copy(), df[~mask], model, value="Human")
new_data.to_csv("data/interim/wildfire_human_reassigned.csv")
print(df.CAUSE_CLASS.value_counts())
new_data.CAUSE_CLASS.value_counts()

/tmp/ipykernel_2006503/2381557319.py:7: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/interim/wildfire_clean.csv")


CAUSE_CLASS
Human      1777492
Natural     326456
Unknown     192368
Name: count, dtype: int64
Reassigned 109084 labeel
CAUSE_CLASS
Human      1777492
Natural     326456
Unknown     192368
Name: count, dtype: int64


CAUSE_CLASS
Human      1883205
Natural     326456
Unknown      86655
Name: count, dtype: int64

In [4]:
from sklearn.preprocessing import LabelEncoder
from src.utils import assign


In [5]:

df = pd.read_csv("data/interim/wildfire_human_reassigned.csv")
model = joblib.load("models/best_sgd_model.pkl")
mask=df["CAUSE_CLASS"].isin(["Human", "Natural"])
new_data = assign(df.copy(), df[~mask], model, encoder={1:"Human", 0:"Natural"})
new_data.to_csv("data/interim/wildfire_natural_reassigned.csv")

/tmp/ipykernel_2006503/934522540.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/interim/wildfire_human_reassigned.csv")


Reassigned 9445 labeel
